# revit-api-rag Pipeline

数据准备阶段 — 在 Colab 中运行

## 流程
1. 克隆项目 & 安装依赖
2. 上传原始数据（API HTML + SDK 代码）
3. 解析 API 文档 → SQLite
4. 解析 SDK 代码 → SQLite
5. Embedding → ChromaDB
6. 下载生成的 .db 文件

## 首次使用 — 按顺序运行所有 Cell

1. **环境准备** — 安装依赖、设置 API Key、配置路径
2. **解压数据** — CHM → api_html，ZIP → sdk_samples
3. **解析数据** — HTML → SQLite，.cs → SQLite
4. **Embedding** — SQLite → ChromaDB 向量库，打包到 Drive
5. **测试 RAG** — 检索 + LLM 生成

## 再次使用 — 只需运行标记为 🔄 的 Cell

1. 🔄 环境准备
2. 🔄 恢复数据（从 Drive 解压 tar.gz）
3. 🔄 测试 RAG

## Step 0: 环境准备

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# 克隆项目
#!git clone https://github.com/imkcrevit/revit-api-rag.git
%cd /content/drive/MyDrive/Colab_Projects/revit-api-rag/

!ls

In [ ]:
# 安装 pipeline 依赖
!pip install -r requirements-pipeline.txt -q

In [3]:
# 验证关键包是否安装成功
import chromadb
import google.genai
import yaml
print("所有关键依赖已就绪 ✅")

所有关键依赖已就绪 ✅


In [6]:
# 设置 API Key（在 Colab 的 Secrets 中添加，不要硬编码）
import os
from google.colab import userdata

os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')  # 如果用 OpenAI

print('API Key 已设置 ✅')

TimeoutException: Requesting secret OPENROUTER_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.

In [ ]:
# 在 Colab 中运行，看看现在的情况
%cd /content/drive/MyDrive/Colab_Projects/revit-api-rag

# 1. 把旧文件移到 legacy/ 目录保留（不删除，以后迁移代码时参考）
!mkdir -p legacy
!mv main.ipynb legacy/
!mv split_revit.ipynb legacy/
!mv selct_all_api.ipynb legacy/
!mv setup.py legacy/
!mv requirements.txt legacy/
!mv union_merge_config.yaml legacy/
!mv output_text_1023.md legacy/
!mv project_dataset.json legacy/
!mv full_api.txt legacy/
!mv deepseek_tokenizer_v3 legacy/
!mv extra_data legacy/
!mv revit_sdk_prund legacy/
!mv revit_sdk_collection legacy/

# 2. 旧数据库移到 data/ 目录（后续还要用）
!mkdir -p data/legacy_db
!mv chromadb0815_api_1.db data/legacy_db/
!mv chromadb1022_code_1.db data/legacy_db/
!mv revit_api.db data/legacy_db/

# 3. 旧图片移到 docs/
!mkdir -p docs/images
!mv *.png docs/images/
!mv *.jpg docs/images/

# 4. 把新骨架从嵌套目录提升到根目录
!cp -r revit-api-rag/pipeline ./
!cp -r revit-api-rag/server ./
!cp -r revit-api-rag/config ./
!cp -r revit-api-rag/scripts ./
!cp revit-api-rag/requirements-pipeline.txt ./
!cp revit-api-rag/requirements-server.txt ./
!cp revit-api-rag/README.md ./README.md

# 5. 删除嵌套目录和压缩包
!rm -rf revit-api-rag/
!rm -f revit-api-rag-skeleton.tar.gz

# 6. 创建数据目录
!mkdir -p data/{chromadb,sqlite,knowledge,raw}

# 7. 验证最终结构
!echo "=== 项目根目录 ==="
!ls
!echo ""
!echo "=== pipeline/ ==="
!ls pipeline/
!echo ""
!echo "=== server/ ==="
!ls server/
!echo ""
!echo "=== legacy/ ==="
!ls legacy/


In [ ]:
%cd /content/drive/MyDrive/Colab_Projects/revit_data/

!ls

/content/drive/MyDrive/Colab_Projects/revit_data
RevitAPI  RevitAPI.chm


In [ ]:
# 复制配置文件
!cp config/config.example.yaml config/config.yaml
print('配置文件已创建 ✅')
print('如需修改 embedding provider，请编辑 config/config.yaml')

配置文件已创建 ✅
如需修改 embedding provider，请编辑 config/config.yaml


In [ ]:
%cd /content/drive/MyDrive/Colab_Projects/revit-api-rag

# 1. 提交当前更改到 colab 分支
!git add -A
!git commit -m "refactor: restructure project - separate pipeline/server/legacy"

# 2. 推送 colab 分支
!git push origin colab

# 3. 切换到 main 分支并合并
!git checkout main
!git merge colab -m "merge: new project structure from colab branch"

# 4. 推送 main
!git push origin main

# 5. 切回 main 继续工作
!git branch

## Step 1: 上传原始数据

In [ ]:
# 方式一：从 Google Drive 挂载
from google.colab import drive
drive.mount('/content/drive')

# 方式二：直接上传文件
# from google.colab import files
# uploaded = files.upload()

解压文件到目标位置

Cell-1 解压chm

In [ ]:
%cd /content/drive/MyDrive/Colab_Projects/revit-api-rag

# 创建目录
!mkdir -p data/raw/api_html
!mkdir -p data/raw/sdk_samples

# 解压 CHM 到 Colab 本地（不是 Drive，速度快很多）
!apt-get install -y p7zip-full -q
!7z x "/content/drive/MyDrive/Colab_Projects/revit_data/RevitAPI.chm" -o/tmp/chm_out -y

# 看看解出来什么
!ls /tmp/chm_out/

/content/drive/MyDrive/Colab_Projects/revit-api-rag
Reading package lists...
Building dependency tree...
Reading state information...
p7zip-full is already the newest version (16.02+dfsg-8).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.

7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,2 CPUs Intel(R) Xeon(R) CPU @ 2.20GHz (406F0),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan /content/drive/MyDrive/Colab_Projects/revit_data/                                                           1 file, 45216149 bytes (44 MiB)

Extracting archive: /content/drive/MyDrive/Colab_Projects/revit_data/RevitAPI.chm
--
Path = /content/drive/MyDrive/Colab_Projects/revit_data/RevitAPI.chm
Type = Chm
Physical Size = 45216149

  0%      0% 277 - html/024faa95-cc70-99fb-a29d

复制API HTML到项目目录

In [ ]:
%cd /content

# 1. 解压到 Colab 本地磁盘（不经过 Drive，非常快）
!mkdir -p /content/api_html
!7z x "/content/drive/MyDrive/Colab_Projects/revit_data/RevitAPI.chm" -o/content/chm_tmp -y
!mv /content/chm_tmp/html/* /content/api_html/
!rm -rf /content/chm_tmp

# 2. 验证
!find /content/api_html/ -name "*.htm*" | wc -l
print('API 解压到本地完成 ✅')

/content

7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,2 CPUs Intel(R) Xeon(R) CPU @ 2.20GHz (406F0),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan /content/drive/MyDrive/Colab_Projects/revit_data/                                                           1 file, 45216149 bytes (44 MiB)

Extracting archive: /content/drive/MyDrive/Colab_Projects/revit_data/RevitAPI.chm
--
Path = /content/drive/MyDrive/Colab_Projects/revit_data/RevitAPI.chm
Type = Chm
Physical Size = 45216149

  0%      1% 420 - html/03a2609a-7777-f049-5013-c90eaa7054a2.htm                                                          2% 718 - html/05fdabfa-c48f-4532-c1fb-27544e5494fd.htm

In [ ]:
# 3. 打包成一个 tar.gz（一个大文件写入 Drive 不会超时）
!tar -czf /content/drive/MyDrive/Colab_Projects/revit_data/api_html.tar.gz -C /content api_html/
print('API 打包完成 ✅')

API 打包完成 ✅


In [ ]:
# 4. SDK 解压到本地
!mkdir -p /content/sdk_samples
!unzip -o -q "/content/drive/MyDrive/Colab_Projects/revit_data/Samples.zip" -d /content/sdk_samples/

# 验证
!find /content/sdk_samples/ -name "*.cs" | wc -l
print('SDK 解压到本地完成 ✅')

1668
SDK 解压到本地完成 ✅


In [ ]:
# 5. SDK 打包
!tar -czf /content/drive/MyDrive/Colab_Projects/revit_data/sdk_samples.tar.gz -C /content sdk_samples/
print('SDK 打包完成 ✅')

SDK 打包完成 ✅


## 第二次运行从此处开始运行，首先检查是否有解压文件，没有解压则开始解压
Cell2 - Colab 重新连接，找到打包的压缩文件\

In [ ]:
# 快速恢复数据（解压 tar.gz 到本地，几秒钟）
!tar -xzf /content/drive/MyDrive/Colab_Projects/revit_data/api_html.tar.gz -C /content/
!tar -xzf /content/drive/MyDrive/Colab_Projects/revit_data/sdk_samples.tar.gz -C /content/
print('数据恢复完成 ✅')

数据恢复完成 ✅


## Step 2: 解析 API 文档

In [ ]:
import sys
sys.path.insert(0,'/content/drive/MyDrive/Colab_Projects/revit-api-rag')

# 重新加载模块（因为之前 import 过旧版本）
import importlib
import pipeline.api_parser.parse_chm as parse_module
importlib.reload(parse_module)
from pipeline.api_parser.parse_chm import parse_all_api_html, save_to_sqlite

# 解析（用本地路径，快）
api_data = parse_all_api_html('/content/api_html/')

# 先看看前3条数据的质量
for item in api_data[:3]:
    print(f"名称: {item['name']}")
    print(f"命名空间: {item['namespace']}")
    print(f"描述: {item['info'][:100]}...")
    print(f"成员数: {len(item['members'].split(chr(10))) if item['members'] else 0}")
    print("---")

# 存入 SQLite
save_to_sqlite(api_data, '/content/drive/MyDrive/Colab_Projects/revit-api-rag/data/sqlite/revit_api.db')
print(f'\nAPI 解析完成：{len(api_data)} 条 ✅')

~删除原有的旧数据库数据，重新保存~

In [ ]:
!rm -f /content/drive/MyDrive/Colab_Projects/revit-api-rag/data/sqlite/revit_api.db

save_to_sqlite(api_data, '/content/drive/MyDrive/Colab_Projects/revit-api-rag/data/sqlite/revit_api.db')
print(f'\nAPI 解析完成：{len(api_data)} 条 ✅')

已保存 28863 条数据到 /content/drive/MyDrive/Colab_Projects/revit-api-rag/data/sqlite/revit_api.db

API 解析完成：28863 条 ✅


## Step 3: 解析 SDK 代码

In [ ]:
# 验证
!find /content/sdk_samples/ -name "*.cs" | wc -l
print('SDK 解压完成 ✅')

1668
SDK 解压完成 ✅


In [ ]:
from pipeline.sdk_parser.extract import extract_all_sdk_projects, save_to_sqlite

sdk_data = extract_all_sdk_projects('/content/sdk_samples/')

save_to_sqlite(sdk_data, '/content/drive/MyDrive/Colab_Projects/revit-api-rag/data/sqlite/revit_sdk.db')

print(f'SDK 解析完成：{len(sdk_data)} 条 ✅')

找到 1 个 SDK 项目
共提取 1668 个 .cs 文件
已保存 1668 条数据到 /content/drive/MyDrive/Colab_Projects/revit-api-rag/data/sqlite/revit_sdk.db
SDK 解析完成：1668 条 ✅


## Step 4: Embedding 向量化

设置APIKEY All From OpenRouter

In [ ]:
import os
from google.colab import userdata

os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
print('OpenRouter API Key 已设置 ✅')

OpenRouter API Key 已设置 ✅


In [ ]:
# 测试 embedding 是否正常
import importlib
import config as config_module
importlib.reload(config_module)
from config import load_config
from pipeline.embedder.providers import create_embedding

config = load_config('/content/drive/MyDrive/Colab_Projects/revit-api-rag/config/config.yaml')
embedder = create_embedding(config)

# 测试一条
test = embedder.embed_query("Create structural column in Revit")
print(f'Model: {embedder.model_name}')
print(f'Dimension: {len(test)}')
print(f'前5个值: {test[:5]}')
print('Embedding 测试成功 ✅')

Model: openai/text-embedding-3-large
Dimension: 3072
前5个值: [-0.0051480429247021675, -0.05087319016456604, -0.018109237775206566, 0.011834426783025265, 0.028094956651329994]
Embedding 测试成功 ✅


开始Embedding

In [ ]:
from config import load_config
from pipeline.embedder.embed import embed_api_data, embed_code_data

config = load_config('/content/drive/MyDrive/Colab_Projects/revit-api-rag/config/config.yaml')
version = config.get('revit_version', '2026')

print(f'Embedding provider: {config["embedding"]["provider"]}')
print(f'Revit version: {version}')
print('开始向量化...')

Embedding provider: openai
Revit version: 2026
开始向量化...


In [ ]:
base = '/content/drive/MyDrive/Colab_Projects/revit-api-rag'

# API 向量化
embed_api_data(
    config=config,
    api_db_path=f'{base}/data/sqlite/revit_api.db',
    chromadb_dir='/content/chromadb_api/',
)
print('API 向量化完成 ✅')

In [ ]:
# SDK 向量化
import importlib
import pipeline.embedder.embed as embed_module
importlib.reload(embed_module)
from pipeline.embedder.embed import embed_code_data

embed_code_data(
    config=config,
    sdk_db_path=f'{base}/data/sqlite/revit_sdk.db',
    chromadb_dir='/content/chromadb_code/',
)
print('SDK 向量化完成 ✅')

从 /content/drive/MyDrive/Colab_Projects/revit-api-rag/data/sqlite/revit_sdk.db 读取 0 条 SDK 代码数据
已写入 /content/chromadb_code/meta.json
Code 向量化完成，共 0 条，存入 /content/chromadb_code/
SDK 向量化完成 ✅


In [ ]:
# 完成后打包到 Drive
!tar -czf {base}/data/chromadb_code.tar.gz -C /content chromadb_code/
print('SDK 向量库已保存到 Drive ✅')

SDK 向量库已保存到 Drive ✅


打包回Drive

In [ ]:
# 打包存回 Drive
!tar -czf {base}/data/chromadb_api.tar.gz -C /content chromadb_api/
!tar -czf {base}/data/chromadb_code.tar.gz -C /content chromadb_code/
print('向量库已保存到 Drive ✅')

向量库已保存到 Drive ✅


## Step 5: 验证 & 下载

In [ ]:
import chromadb
import json

paths = {
    'api': '/content/chromadb_api/',
    'code': '/content/chromadb_code/',
}

for db_type, db_dir in paths.items():
    meta_path = f'{db_dir}/meta.json'
    try:
        with open(meta_path) as f:
            meta = json.load(f)
        print(f'\n{db_type.upper()} 向量库:')
        print(f'  Provider: {meta["embedding_provider"]}')
        print(f'  Model: {meta["embedding_model"]}')
        print(f'  Dimension: {meta["embedding_dimension"]}')
        print(f'  Records: {meta["record_count"]}')

        client = chromadb.PersistentClient(path=db_dir)
        for col in client.list_collections():
            print(f'  Collection: {col.name}, Count: {col.count()}')
    except FileNotFoundError:
        print(f'\n{db_type.upper()} 向量库: 未找到，需要重新生成')


API 向量库:
  Provider: openai
  Model: openai/text-embedding-3-large
  Dimension: 3072
  Records: 28863
  Collection: revit_api, Count: 28863

CODE 向量库:
  Provider: openai
  Model: openai/text-embedding-3-large
  Dimension: 3072
  Records: 1668
  Collection: revit_sdk, Count: 1668


In [ ]:
# 打包数据文件用于下载
!tar -czf revit_rag_data.tar.gz data/
print('数据已打包: revit_rag_data.tar.gz')
print(f'文件大小: {os.path.getsize("revit_rag_data.tar.gz") / 1024 / 1024:.1f} MB')

# 下载
from google.colab import files
files.download('revit_rag_data.tar.gz')

# 或者保存到 Google Drive
# !cp revit_rag_data.tar.gz /content/drive/MyDrive/

数据已打包: revit_rag_data.tar.gz
文件大小: 0.0 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 测试数据库db文件准确度

In [ ]:
# Cell 1: 加载向量库 + 配置
import chromadb
from pipeline.embedder.providers import create_embedding
from config import load_config

base = '/content/drive/MyDrive/Colab_Projects/revit-api-rag'
config = load_config(f'{base}/config/config.yaml')
embedder = create_embedding(config)

# 加载 ChromaDB
api_client = chromadb.PersistentClient(path='/content/chromadb_api/')
code_client = chromadb.PersistentClient(path='/content/chromadb_code/')
api_collection = api_client.get_collection("revit_api")
code_collection = code_client.get_collection("revit_sdk")

print(f"API 向量库: {api_collection.count()} 条")
print(f"Code 向量库: {code_collection.count()} 条")
print("加载完成 ✅")

In [ ]:
# Cell 2: 检索函数
def search_rag(query: str, api_top_k: int = 10, code_top_k: int = 5):
    """向量检索"""
    query_embedding = embedder.embed_query(query)

    api_results = api_collection.query(
        query_embeddings=[query_embedding],
        n_results=api_top_k,
    )

    code_results = code_collection.query(
        query_embeddings=[query_embedding],
        n_results=code_top_k,
    )

    return api_results, code_results


def format_results(api_results, code_results):
    """格式化检索结果"""
    print("=" * 60)
    print("📖 API 检索结果：")
    print("=" * 60)
    for i, (doc, meta, dist) in enumerate(zip(
        api_results['documents'][0],
        api_results['metadatas'][0],
        api_results['distances'][0]
    )):
        print(f"\n[{i+1}] 相似度: {1-dist:.4f}")
        print(f"    名称: {meta.get('name', '')}")
        print(f"    内容: {doc[:150]}...")

    print("\n" + "=" * 60)
    print("💻 代码检索结果：")
    print("=" * 60)
    for i, (doc, meta, dist) in enumerate(zip(
        code_results['documents'][0],
        code_results['metadatas'][0],
        code_results['distances'][0]
    )):
        print(f"\n[{i+1}] 相似度: {1-dist:.4f}")
        print(f"    项目: {meta.get('project', '')} / {meta.get('filename', '')}")
        print(f"    代码: {doc[:200]}...")

In [ ]:
# Cell 3: 测试检索
query = "创建结构柱"
api_results, code_results = search_rag(query)
format_results(api_results, code_results)

In [ ]:
# Cell 4: LLM 生成（通过 OpenRouter）
from openai import OpenAI
import os

llm_client = OpenAI(
    api_key=os.environ['OPENROUTER_API_KEY'],
    base_url="https://openrouter.ai/api/v1",
)

def generate_code(query: str, api_results, code_results, model: str = "google/gemini-2.5-flash"):
    """RAG 生成：检索结果 + LLM 生成代码"""

    # 拼接检索到的 API 参考
    api_context = "\n\n".join([
        f"API: {meta.get('name', '')}\n{doc[:300]}"
        for doc, meta in zip(api_results['documents'][0], api_results['metadatas'][0])
    ])

    # 拼接检索到的代码参考
    code_context = "\n\n".join([
        f"// File: {meta.get('filename', '')}\n{doc[:500]}"
        for doc, meta in zip(code_results['documents'][0], code_results['metadatas'][0])
    ])

    system_prompt = """You are a professional BIM engineer expert in Revit API.
You write C# code for Revit plugins following these standards:
1. Completeness: Include the entire process from start to finish
2. Professionalism: Correctly handle Revit element characteristics
3. Robustness: Include error handling and boundary condition checking
4. Scalability: Easy to extend
5. Best practice: Follow Revit API development specifications

Give the user a complete, working C# plugin code solution."""

    user_prompt = f"""User question: {query}

Revit API Reference:
{api_context}

Code Reference:
{code_context}

Based on the references above, generate a complete Revit C# plugin to solve the user's question.
Include all necessary using statements, the IExternalCommand class, and proper Transaction handling."""

    response = llm_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.3,
        max_tokens=4096,
    )

    return response.choices[0].message.content


# 运行完整 RAG 流程
query = "创建结构柱"
print(f"🔍 查询: {query}\n")

# 检索
api_results, code_results = search_rag(query, api_top_k=15, code_top_k=5)
print(f"检索到 API: {len(api_results['documents'][0])} 条, Code: {len(code_results['documents'][0])} 条")

# 生成
print("\n🤖 生成中...\n")
answer = generate_code(query, api_results, code_results)
print(answer)

## 完成！

下一步：
1. 将 `revit_rag_data.tar.gz` 上传到 GCP 服务器
2. 解压到项目的 `data/` 目录
3. 运行 `python -m server.app.main` 启动服务